In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import wandb
import os
import cv2
import pandas as pd
import json


In [11]:

class CustomDataset(Dataset):
    def __init__(self, images_dir, annotations, transforms=None):
        """
        images_dir: path to 'train' folder
        annotations: dictionary from coco.json
                     keys = relative paths to images, e.g., 'video1/img1/frame0001.jpg'
        """
        self.images_dir = images_dir
        self.annotations = annotations
        self.transforms = transforms
        self.image_names = list(annotations.keys())

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]  # e.g., 'video1/img1/frame0001.jpg'
        img_path = os.path.join(self.images_dir, img_name)

        # Safety check
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"Image not found: {img_path}")

        img = cv2.imread(img_path)
        if img is None:
            raise ValueError(f"Failed to read image: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = torch.as_tensor(img / 255., dtype=torch.float32).permute(2,0,1)

        # Load annotations
        ann = self.annotations[img_name]
        boxes = torch.as_tensor(ann['boxes'], dtype=torch.float32)
        labels = torch.as_tensor(ann['labels'], dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([idx])}

        if self.transforms:
            img = self.transforms(img)

        return img, target

In [3]:
def get_fasterrcnn_model(num_classes=2, pretrained=True):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=pretrained)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

In [4]:
wandb.init(project="fasterrcnn-phase1", name="baseline-resnet50-fpn")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: saniav2711 (ishaya-2716) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
annotations_file = "/content/drive/MyDrive/Project/coco_train.json"
with open(annotations_file, "r") as f:
    annotations = json.load(f)


In [7]:
images_dir = "/content/drive/MyDrive/Project/train"
dataset = CustomDataset(images_dir, annotations)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))


In [13]:
images_dir = "/content/drive/MyDrive/Project/train"

# Load COCO JSON
import json
with open("/content/drive/MyDrive/Project/coco_train.json", "r") as f:
    annotations = json.load(f)

dataset = CustomDataset(images_dir, annotations)

In [14]:
from torch.utils.data import DataLoader

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=lambda x: tuple(zip(*x))  # required for variable-length targets
)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_fasterrcnn_model(num_classes=3)
model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 221MB/s]


FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

In [9]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)


In [15]:
from torch.utils.data import random_split

dataset_size = len(dataset)
train_size = int(0.8 * dataset_size)
val_size = dataset_size - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [16]:
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=lambda x: tuple(zip(*x))
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,   # no need to shuffle validation
    collate_fn=lambda x: tuple(zip(*x))
)

In [17]:
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for imgs, targets in train_loader:
        imgs = list(img.to(device) for img in imgs)
        targets = [{k: v.to(device) for k,v in t.items()} for t in targets]

        loss_dict = model(imgs, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {avg_loss:.4f}")
    wandb.log({"epoch": epoch+1, "train_loss": avg_loss})

    # Optional: validation loss
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs = list(img.to(device) for img in imgs)
            targets = [{k: v.to(device) for k,v in t.items()} for t in targets]
            loss_dict = model(imgs, targets)
            losses = sum(loss for loss in loss_dict.values())
            val_loss += losses.item()

    avg_val_loss = val_loss / len(val_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Val Loss: {avg_val_loss:.4f}")
    wandb.log({"epoch": epoch+1, "val_loss": avg_val_loss})

FileNotFoundError: Image not found: /content/drive/MyDrive/Project/train/annotations

In [ ]:
torch.save(model.state_dict(), "fasterrcnn_phase1.pth")
wandb.save("fasterrcnn_phase1.pth")
print("✅ Training complete, model saved.")


Coding again

In [29]:
import json, re
from collections import defaultdict

coco_path = "/content/drive/MyDrive/coco_train.json"

with open(coco_path, 'r') as f:
    content = f.read()

# Find the last complete annotation (last full closing brace before truncation)
last_complete = content.rfind('}')  # last full object
salvaged = content[:last_complete+1]  # trim the broken tail

# The annotations array is now unclosed — fix the JSON structure
# Close the annotations array and root object
salvaged = salvaged + ']}'

try:
    data = json.loads(salvaged)
    print(f"✅ Salvaged! Images: {len(data.get('images', []))}, Annotations: {len(data.get('annotations', []))}")
except json.JSONDecodeError as e:
    print(f"Still broken at {e.pos}, trying deeper fix...")
    # If it failed, the structure may need more closing — check what top-level keys exist
    print("Top of file:", content[:500])

Still broken at 13751708, trying deeper fix...
Top of file: {"images": [{"id": 0, "file_name": "train/MOT17-02-DPM/img1/000001.jpg", "width": 1920, "height": 1080}, {"id": 1, "file_name": "train/MOT17-02-DPM/img1/000002.jpg", "width": 1920, "height": 1080}, {"id": 2, "file_name": "train/MOT17-02-DPM/img1/000003.jpg", "width": 1920, "height": 1080}, {"id": 3, "file_name": "train/MOT17-02-DPM/img1/000004.jpg", "width": 1920, "height": 1080}, {"id": 4, "file_name": "train/MOT17-02-DPM/img1/000005.jpg", "width": 1920, "height": 1080}, {"id": 5, "file_name": 


In [30]:
import json, re
from collections import defaultdict

coco_path = "/content/drive/MyDrive/Project/coco_train.json"

with open(coco_path, 'r') as f:
    content = f.read()

# Find the last complete annotation (last full closing brace before truncation)
last_complete = content.rfind('}')  # last full object
salvaged = content[:last_complete+1]  # trim the broken tail

# The annotations array is now unclosed — fix the JSON structure
# Close the annotations array and root object
salvaged = salvaged + ']}'

try:
    data = json.loads(salvaged)
    print(f"✅ Salvaged! Images: {len(data.get('images', []))}, Annotations: {len(data.get('annotations', []))}")
except json.JSONDecodeError as e:
    print(f"Still broken at {e.pos}, trying deeper fix...")
    # If it failed, the structure may need more closing — check what top-level keys exist
    print("Top of file:", content[:500])

Still broken at 4194304, trying deeper fix...
Top of file: {"images": [{"id": 0, "file_name": "train/MOT17-02-DPM/img1/000001.jpg", "width": 1920, "height": 1080}, {"id": 1, "file_name": "train/MOT17-02-DPM/img1/000002.jpg", "width": 1920, "height": 1080}, {"id": 2, "file_name": "train/MOT17-02-DPM/img1/000003.jpg", "width": 1920, "height": 1080}, {"id": 3, "file_name": "train/MOT17-02-DPM/img1/000004.jpg", "width": 1920, "height": 1080}, {"id": 4, "file_name": "train/MOT17-02-DPM/img1/000005.jpg", "width": 1920, "height": 1080}, {"id": 5, "file_name": 


In [23]:
!pip install pycocotools

In [19]:
!pip install ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 6.7 MB/s eta 0:00:00
